In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [14]:
corpus = [
    "the cat loves the mouse",
    "the dog chases the cat",
    "machine learning is amazing"
]

In [15]:
tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words='english',
    max_features=1000
)

In [16]:
tfidf_matrix = tfidf_vectorizer.fit_transform(corpus)

In [20]:
tfidf_matrix.shape

(3, 8)

In [23]:
print("Vocabulary:", tfidf_vectorizer.get_feature_names_out())
print("TF-IDF Matrix:")
print(tfidf_matrix.toarray())

Vocabulary: ['amazing' 'cat' 'chases' 'dog' 'learning' 'loves' 'machine' 'mouse']
TF-IDF Matrix:
[[0.         0.4736296  0.         0.         0.         0.62276601
  0.         0.62276601]
 [0.         0.4736296  0.62276601 0.62276601 0.         0.
  0.         0.        ]
 [0.57735027 0.         0.         0.         0.57735027 0.
  0.57735027 0.        ]]


In [24]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

# Sample corpus
corpus = [
    "machine learning is amazing",
    "deep learning uses neural networks",
    "machine learning and deep learning are both AI",
    "natural language processing is cool"
]

# Fit TF-IDF
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(corpus)

# Get feature names (vocabulary)
feature_names = tfidf.get_feature_names_out()
print("Vocabulary:", feature_names)

# Method 1A: Get scores for all words in a specific document
def get_word_scores_document(tfidf_matrix, feature_names, doc_index):
    """Get all word scores for a specific document"""
    doc_scores = tfidf_matrix[doc_index].toarray()[0]
    word_scores = list(zip(feature_names, doc_scores))
    # Filter out zero scores and sort
    word_scores = [(word, score) for word, score in word_scores if score > 0]
    return sorted(word_scores, key=lambda x: x[1], reverse=True)

# Get scores for document 0
print("\n=== Word Scores for Document 0 ===")
scores_doc0 = get_word_scores_document(tfidf_matrix, feature_names, 0)
for word, score in scores_doc0:
    print(f"{word}: {score:.4f}")

# Method 1B: Get score for specific word in specific document
def get_specific_word_score(tfidf_matrix, feature_names, doc_index, target_word):
    """Get TF-IDF score for a specific word in a specific document"""
    if target_word not in feature_names:
        return f"Word '{target_word}' not in vocabulary"

    word_index = list(feature_names).index(target_word)
    score = tfidf_matrix[doc_index].toarray()[0][word_index]
    return score

# Get specific word scores
print("\n=== Specific Word Scores ===")
print(f"'machine' in doc0: {get_specific_word_score(tfidf_matrix, feature_names, 0, 'machine'):.4f}")
print(f"'learning' in doc0: {get_specific_word_score(tfidf_matrix, feature_names, 0, 'learning'):.4f}")
print(f"'deep' in doc0: {get_specific_word_score(tfidf_matrix, feature_names, 0, 'deep'):.4f}")

Vocabulary: ['ai' 'amazing' 'cool' 'deep' 'language' 'learning' 'machine' 'natural'
 'networks' 'neural' 'processing' 'uses']

=== Word Scores for Document 0 ===
amazing: 0.7020
machine: 0.5535
learning: 0.4481

=== Specific Word Scores ===
'machine' in doc0: 0.5535
'learning' in doc0: 0.4481
'deep' in doc0: 0.0000


In [25]:
# To get TF and IDF components separately, we need custom implementation
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer

def detailed_tfidf_analysis(corpus):
    """Get detailed TF, IDF, and TF-IDF scores"""

    # First get CountVectorizer for TF
    count_vec = CountVectorizer(stop_words='english')
    tf_matrix = count_vec.fit_transform(corpus)
    feature_names = count_vec.get_feature_names_out()

    # Calculate TF (Term Frequency)
    tf = tf_matrix.toarray()

    # Calculate IDF manually
    n_docs = len(corpus)
    df = np.count_nonzero(tf, axis=0)  # Document frequency
    idf = np.log(n_docs / (df + 1)) + 1  # Smooth IDF

    # Calculate TF-IDF
    tfidf_scores = tf * idf

    return feature_names, tf, idf, tfidf_scores

# Get detailed analysis
feature_names, tf, idf, tfidf_scores = detailed_tfidf_analysis(corpus)

print("\n=== Detailed TF-IDF Analysis ===")
print("Feature names:", feature_names)
print("\nTF (Term Frequency) matrix:")
print(tf)
print("\nIDF (Inverse Document Frequency) values:")
for word, idf_val in zip(feature_names, idf):
    print(f"{word}: {idf_val:.4f}")
print("\nTF-IDF Scores:")
print(tfidf_scores)


=== Detailed TF-IDF Analysis ===
Feature names: ['ai' 'amazing' 'cool' 'deep' 'language' 'learning' 'machine' 'natural'
 'networks' 'neural' 'processing' 'uses']

TF (Term Frequency) matrix:
[[0 1 0 0 0 1 1 0 0 0 0 0]
 [0 0 0 1 0 1 0 0 1 1 0 1]
 [1 0 0 1 0 2 1 0 0 0 0 0]
 [0 0 1 0 1 0 0 1 0 0 1 0]]

IDF (Inverse Document Frequency) values:
ai: 1.6931
amazing: 1.6931
cool: 1.6931
deep: 1.2877
language: 1.6931
learning: 1.0000
machine: 1.2877
natural: 1.6931
networks: 1.6931
neural: 1.6931
processing: 1.6931
uses: 1.6931

TF-IDF Scores:
[[0.         1.69314718 0.         0.         0.         1.
  1.28768207 0.         0.         0.         0.         0.        ]
 [0.         0.         0.         1.28768207 0.         1.
  0.         0.         1.69314718 1.69314718 0.         1.69314718]
 [1.69314718 0.         0.         1.28768207 0.         2.
  1.28768207 0.         0.         0.         0.         0.        ]
 [0.         0.         1.69314718 0.         1.69314718 0.
  0.       